# 07 Gradio 课程交互演示

**目标：** 在 Notebook 中启动本地图片分类演示，显示模型类别概率与可选 Grad-CAM。

**前置条件：** 有一份经验证选定的真实训练权重；已安装 `requirements-demo.txt`；主实验与报告评价已完成。

**输入：** `DEMO_RUN` 指定的 `best.pt`、上传的叶片图片；可选 `demo/examples/` 示例图片。

**输出：** 本地 Gradio 交互界面。默认不启动服务，不调用外网分享。

界面显示模型预测，置信度表示模型输出概率；真实场景可能存在明显误判。养护信息只使用经确认的材料，`demo/disease_info.json` 为空时不展示建议。

In [ ]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

## 1. 选择演示模型

填写真实的实验目录名，模型的类别顺序与输入预处理从保存的 checkpoint 读取。示例图片可在后期放入 `demo/examples/`，不要将整个测试集作为界面示例。

In [ ]:
LAUNCH_DEMO = False
DEMO_RUN = ""  # 填写 outputs/runs/ 下最终选定的实验目录名

demo_run_dir = PROJECT_ROOT / "outputs" / "runs" / DEMO_RUN if DEMO_RUN else None
demo_ready = bool(demo_run_dir and (demo_run_dir / "best.pt").is_file())
if not DEMO_RUN:
    print("请先填写 DEMO_RUN，选择已完成训练的演示模型。")
elif not demo_ready:
    print("所选实验尚无 best.pt，请核对目录名并完成训练。")
else:
    print("模型已选择：", demo_run_dir)
print("是否启动演示：", LAUNCH_DEMO)

## 2. 启动本地界面

只有开启 `LAUNCH_DEMO` 且权重存在时才导入演示依赖、加载模型并启动。可选 Grad-CAM 在勾选后计算。首次运行前先完成模型检查；关闭界面时可以运行后面的停止单元格。

In [ ]:
demo = None
if not LAUNCH_DEMO:
    print("尚未启动演示。配置好模型后，将 LAUNCH_DEMO 设为 True。")
elif not demo_ready:
    print("缺少演示模型，暂不能启动。")
else:
    import gradio as gr
    from src.data import build_transforms
    from src.evaluate import load_trained_model, predict_image
    from src.explain import gradcam_image
    from src.utils import get_device

    device = get_device(BASE.get("device", "auto"))
    model, checkpoint = load_trained_model(demo_run_dir, device=device)
    data_config = checkpoint["config"]["data"]
    transform = build_transforms("none", **{k: data_config[k]
                                 for k in ("image_size", "mean", "std") if k in data_config})
    class_names = checkpoint["class_names"]
    display_names = {item["label"]: item.get("display_name", item["label"])
                     for item in LABELS.get("classes", [])}
    disease_path = PROJECT_ROOT / "demo" / "disease_info.json"
    disease_info = load_config(disease_path) if disease_path.exists() else {}
    disease_details = {item["label"]: item for item in disease_info.get("classes", [])
                       if isinstance(item, dict) and item.get("label")}

    def predict_for_demo(image, show_heatmap):
        if image is None:
            return {}, None, "请先上传一张叶片图片。"
        result = predict_image(model, image, transform, class_names, device=device)
        probabilities = {display_names.get(k, k): v for k, v in result["probabilities"].items()}
        overlay = None
        if show_heatmap:
            overlay = gradcam_image(model, checkpoint["model_name"], image, transform,
                                    target_class=result["label_index"], device=device)
        label_text = display_names.get(result["label"], result["label"])
        details = f"**模型预测：{label_text}**，输出概率 {result['confidence']:.1%}。"
        details += "\n\n用于课程研究展示；真实场景误差需结合外部测试结果判断。"
        extra = disease_details.get(result["label"])
        if isinstance(extra, dict) and extra.get("reference_text"):
            details += "\n\n" + str(extra["reference_text"])
        return probabilities, overlay, details

    example_dir = PROJECT_ROOT / "demo" / "examples"
    example_paths = sorted(p for p in example_dir.glob("*")
                           if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})[:6]
    demo = gr.Interface(
        fn=predict_for_demo,
        inputs=[gr.Image(type="pil", label="上传叶片照片"),
                gr.Checkbox(value=False, label="生成 Grad-CAM 热力图")],
        outputs=[gr.Label(num_top_classes=3, label="类别概率"),
                 gr.Image(type="pil", label="Grad-CAM"), gr.Markdown()],
        title="植物叶片病害识别：迁移学习课程演示",
        description="上传图片查看模型预测；模型类别范围以已训练数据为限。",
        examples=[[str(path), False] for path in example_paths] or None,
    )
    demo.launch(inline=True, share=False)

In [ ]:
# 需要关闭演示时，将 CLOSE_DEMO 改为 True 后单独运行本单元格。
CLOSE_DEMO = False
if CLOSE_DEMO and demo is not None:
    demo.close()
    print("演示已关闭。")

## 结论记录

- 演示模型与选择依据：待填写。
- 已验证的上传、预测、概率显示及热力图功能：待实际演示后填写。
- 示例来源与使用权限：待填写。
- 演示中发现的失败场景及限制：待填写。